In [4]:
import pathlib

import numpy as np
import pandas as pd

In [5]:
from ete3 import Tree

In [6]:
from splitter import RandomSplitter, LargeTreeTraverseOOCSplitter

## Read phenotype and feature data

Get X and y 

In [7]:
data_folder = pathlib.Path("../../data/processed/biolog/")
pheno_file = data_folder / "phenotypes/leaf_phenotypes.tsv"
# NOTE: We are using the reduced version of the kofam features (with correlated features removed)
feature_file = data_folder / "features_reduced/kofam/kofam_leaf_features_reduced.tsv"
pheno_file.is_file(), feature_file.is_file()

(True, True)

In [8]:
from trait_prediction.main import PhenotypeSet, PhenotypeIndex

In [9]:
phenotypeset = PhenotypeSet.read_data(pheno_file)
phenotypeset

PhenotypeSet (n=44)

In [10]:
# Get the y
phenotype_index = PhenotypeIndex(name="Histidine", category="leaf_biolog")
phenotype = phenotypeset[phenotype_index]

In [11]:
from trait_prediction.utils import read_generic_features

In [12]:
# Get X
feature = read_generic_features(
    feature_file, bool_conversion=True, dtype="uint8"
)

In [13]:
common_index = list(set(phenotype.phenotype_data.index) & set(feature.index))
y = phenotype.phenotype_data.loc[common_index]
X = feature.loc[common_index]

In [14]:
X

,K09681,K11921,K04761,K00241,K22310,K23107,K01179,K06400,K07492,K08166,...,K12282,K10924,K12287,K20966,K20922,K26913,K21712,K18678,K21252,K12954
genomeID,,,,,,,,,,,,,,,,,,,,,
GCF_001422405.1,0,0,1,1,1,1,0,1,0,0,...,0,0,0,0,0,0,0,0,1,0
GCF_001422165.1,0,0,1,0,1,1,0,0,1,0,...,0,0,0,0,0,1,0,0,0,0
GCF_001424185.1,0,0,0,0,0,1,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
GCF_001421605.1,0,0,1,1,0,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
GCF_001421665.1,0,0,1,1,0,1,0,1,1,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GCF_001421535.1,0,0,1,1,0,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
GCF_001423125.1,0,0,1,1,1,0,1,1,0,0,...,1,1,1,0,0,0,0,0,0,0
GCF_003258395.1,0,1,1,1,1,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [15]:
y

genomeID
GCF_001422405.1    1
GCF_001422165.1    0
GCF_001424185.1    0
GCF_001421605.1    0
GCF_001421665.1    1
                  ..
GCF_001421535.1    0
GCF_001423125.1    1
GCF_003258395.1    1
GCF_001425445.1    0
GCF_001424045.1    1
Name: Histidine--leaf_biolog, Length: 202, dtype: uint8

In [16]:
# Do all the genomes start with GCF?
[i for i in y.index if not i.startswith("GCF")]

[]

## Read the tree and create train/test split

Create X_train, X_test, y_train, y_test

In [17]:
tree_file = pathlib.Path("../../data/processed/biolog/phylogeny/v2_all_combined_trimmed_tree-labels.newick")
tree_file.is_file()

True

In [18]:
tree = Tree(str(tree_file), format=1)
tree

Tree node '' (0x7fc50fc71ad)

In [19]:
# Get the number of leaves in the tree
len(tree)

1361

In [20]:
# Drop all leaves from tree that do not start with GCF
# Also remove the .fna suffix from the ones that do start with GCF
for leaf in tree:
    if leaf.name.startswith("GCF"):
        leaf.name = leaf.name.rstrip(".fna")
    else:
        leaf.delete()

In [21]:
# new number of leaves in tree
len(tree)

186

In [22]:
# Are all the leaves in our tree present in common_index?
set([leaf.name for leaf in tree]) <= set(common_index)

False

In [23]:
# remove leaves that are not in the common_index
tree_leaves_not_in_index = set([leaf.name for leaf in tree]) - set(common_index)
tree_leaves_not_in_index

{'GCF_001422615.1',
 'GCF_001422765.1',
 'GCF_001423695.1',
 'GCF_001424685.1',
 'GCF_916858675.1',
 'GCF_920984685.1'}

In [24]:
# remove leaves that are not in the common_index
for leaf in tree_leaves_not_in_index:
    tree.search_nodes(name=leaf)[0].delete()

len(tree)

180

In [25]:
final_common_index = set([leaf.name for leaf in tree])

### Random split

In [26]:
random_splitter = RandomSplitter(test_set_ratio=0.2)
samples = list(final_common_index)
test_samples_rs = random_splitter.split(samples)
train_samples_rs = list(set(samples) - set(test_samples_rs))

In [27]:
X_train_rs = X.loc[train_samples_rs]
X_test_rs = X.loc[test_samples_rs]
y_train_rs = y.loc[train_samples_rs]
y_test_rs = y.loc[test_samples_rs]
print(X_train_rs.shape, X_test_rs.shape, y_train_rs.shape, y_test_rs.shape)

(144, 3894) (36, 3894) (144,) (36,)


### Out-of-clade split

In [28]:
ooc_splitter = LargeTreeTraverseOOCSplitter(
    tree,
    test_set_range=(0.2, 0.3),
    single_clades=None,
    n_max_clade=2,
    prefer_small_clade=False,
    growth_data=None,
    min_zeros=0,
    min_ones=0,
    time_out_iter=None,
)
samples = list(final_common_index)
test_samples_ooc = ooc_splitter.split(samples)
train_samples_ooc = list(set(samples) - set(test_samples_ooc))

## Is the OOC split choosing different samples each time?

In [29]:
import random
from functools import reduce

def get_clades(single_clades, test_set_range, samples):
    while True:
        clades = random.sample(single_clades, 2)
        test_samples = reduce(np.union1d, clades)
        if (len(test_samples) / len(samples)) < test_set_range[0] or (
            len(test_samples) / len(samples)
        ) > test_set_range[1]:
            continue
        else:
            break
    return test_samples


In [44]:
test_set_range = (0.2, 0.3)
single_clades = ooc_splitter.compute_single_clades(tree, samples)

print(f"Number of single clades = {len(single_clades)}")

clades_1 = get_clades(single_clades, test_set_range, samples)
clades_2 = get_clades(single_clades, test_set_range, samples)
common_elements = set(clades_1) & set(clades_2)
perc_common = len(common_elements) / min(len(clades_1), len(clades_2)) * 100
print(f"Percentage of common elements = {perc_common}")

Number of single clades = 173
Percentage of common elements = 0.0


Yes, sometimes it does.

In [28]:
X_train_ooc = X.loc[train_samples_ooc]
X_test_ooc = X.loc[test_samples_ooc]
y_train_ooc = y.loc[train_samples_ooc]
y_test_ooc = y.loc[test_samples_ooc]
print(X_train_ooc.shape, X_test_ooc.shape, y_train_ooc.shape, y_test_ooc.shape)

(144, 3894) (36, 3894) (144,) (36,)


## Visualizing the split

## Machine learning performance

In [ ]:
from catboost import CatBoostClassifier
def make_classifier(random_state: int, categorical_feature_names: list[str] | None):
    """
    Creates a classifier.

    Parameters
    ---------
    random_state : int
        Random state.
    categorical_feature_names : list[str] | None
        List of categorical feature names.

    Returns
    ------
    Classifier object.
    """
    clf = CatBoostClassifier(
        # iterations=1000,
        # depth=8,
        # learning_rate=0.03,
        # l2_leaf_reg=3,
        # bootstrap_type="Bayesian",
        # bagging_temperature=1,
        random_state=random_state,
        objective="Logloss",
        cat_features=categorical_feature_names,
        verbose=False,
        allow_writing_files=False,
        thread_count=1,
    )
    return clf


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)
def get_scores(
    y_true: pd.Series, y_pred: pd.Series, index_name: str
) -> pd.DataFrame:
    """
    Calculates the scores for the given true and predicted labels.

    Parameters
    ---------
    y_true : pd.DataFrame
        True labels.
    y_pred : pd.DataFrame
        Predicted labels.

    Returns
    ------
    pd.DataFrame
        Scores.
    """
    scores = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "matthews_corrcoef": matthews_corrcoef(y_true, y_pred),
    }
    return pd.DataFrame(scores, index=[index_name])


In [ ]:
categorical_feature_names = list(X.columns)
clf_rs = make_classifier(42, categorical_feature_names)
clf_ooc = make_classifier(42, categorical_feature_names)

### Random split performance

In [ ]:
clf_rs.fit(X_train_rs, y_train_rs)
y_predict_rs = clf_rs.predict(X_test_rs)

In [ ]:
scores_rs = get_scores(y_test_rs, y_predict_rs, "random split")
scores_rs

### OOC split performance

In [ ]:
clf_ooc.fit(X_train_ooc, y_train_ooc)
y_predict_ooc = clf_ooc.predict(X_test_ooc)

In [ ]:
scores_ooc = get_scores(y_test_ooc, y_predict_ooc, "ooc split")
scores_ooc

## Features

In [ ]:
# Get the top 10 most important features
clf_rs.get_feature_importance(prettified=True)[:10]

In [ ]:
clf_ooc.get_feature_importance(prettified=True)[:10]